# Classificação de tráfego DDoS com PySpark + ClickHouse

Treino de um classificador binário **BENIGN vs ATAQUE** sobre o dataset
[CICDDoS2019](https://www.unb.ca/cic/datasets/ddos-2019.html), lendo os dados
**direto do ClickHouse** via JDBC — nenhum CSV local é usado.

O banco tem 18 tabelas raw (`data_*` / `data2_*`, 70,4 M linhas). A função
`merge()` do ClickHouse expõe todas como se fossem uma tabela só, então a
amostragem e a conversão de tipos acontecem no servidor e só o que interessa
trafega até o Spark.

**Pré-requisitos:** `.env` na raiz do repositório (veja `.env.example`), Java 17+
e as dependências de `requirements.txt`.

## 1. Configuração

In [1]:
import os
import urllib.request
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()


def load_env(path: Path) -> dict:
    """Lê o .env sem depender de python-dotenv."""
    values = {}
    for line in path.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, _, value = line.partition("=")
        values[key.strip()] = value.strip().strip('"').strip("'")
    return values


env_path = REPO_ROOT / ".env"
if not env_path.exists():
    raise FileNotFoundError(f"Crie o .env em {env_path} a partir do .env.example")

env = {**load_env(env_path), **os.environ}

CH_HOST = env["CLICKHOUSE_HOST"]
CH_PORT = env["CLICKHOUSE_PORT"]
CH_USER = env["CLICKHOUSE_USER"]
CH_PASSWORD = env["CLICKHOUSE_PASSWORD"]
CH_DATABASE = env["CLICKHOUSE_DATABASE"]

# A Railway expõe o ClickHouse apenas por HTTPS na 443; a 8123 não é publicada.
JDBC_URL = f"jdbc:clickhouse://{CH_HOST}:{CH_PORT}/{CH_DATABASE}?ssl=true"

print(f"ClickHouse: {CH_HOST}:{CH_PORT}/{CH_DATABASE} (usuário {CH_USER})")

ClickHouse: clickhouse-production-e204.up.railway.app:443/railway (usuário clickhouse)


### Driver JDBC

Usamos o jar `shaded-all`, que empacota o driver junto com o cliente HTTP. O
classifier importa: o `-all` da linha 0.6.x **não** traz `com.clickhouse.client`
e o Spark quebra com `NoClassDefFoundError` na hora de registrar o driver.

O download vai para `jars/` (ignorado pelo Git) em vez de `spark.jars.packages`
porque as coordenadas Ivy do Spark não aceitam classifier.

In [2]:
CLICKHOUSE_JDBC_VERSION = "0.8.6"
JDBC_JAR = REPO_ROOT / "jars" / f"clickhouse-jdbc-{CLICKHOUSE_JDBC_VERSION}-shaded-all.jar"

if not JDBC_JAR.exists():
    JDBC_JAR.parent.mkdir(parents=True, exist_ok=True)
    url = (
        "https://repo1.maven.org/maven2/com/clickhouse/clickhouse-jdbc/"
        f"{CLICKHOUSE_JDBC_VERSION}/{JDBC_JAR.name}"
    )
    print(f"Baixando {JDBC_JAR.name}...")
    urllib.request.urlretrieve(url, JDBC_JAR)

print(f"{JDBC_JAR.name}: {JDBC_JAR.stat().st_size / 1024**2:.1f} MiB")

clickhouse-jdbc-0.8.6-shaded-all.jar: 12.4 MiB


## 2. Sessão Spark

In [3]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder.appName("ddos-classification")
    .master("local[*]")
    .config("spark.jars", str(JDBC_JAR))
    .config("spark.driver.memory", "2g")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

print(f"Spark {spark.version}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/09/02 13:31:30 WARN Utils: Your hostname, afonsolelis-Latitude-3450, resolves to a loopback address: 127.0.1.1; using 10.128.128.171 instead (on interface wlp0s20f3)
26/09/02 13:31:30 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


26/09/02 13:31:30 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


/home/afonsolelis/repos/clickhouse-railway/.venv/lib/python3.12/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Spark 4.2.0


## 3. Extração dos dados

A camada raw guarda **todas as 88 colunas como `String`** — foi um carregamento
fiel ao CSV, sem tipagem. A conversão para `Float64` acontece aqui, na query,
para que o Spark já receba números.

Dois cuidados na query:

- **`isFinite`** — colunas como `flow_bytes_s` contêm `Infinity`/`NaN` quando a
  duração do fluxo é zero. Valores não-finitos (e strings não numéricas) viram
  `0.0`, o que evita `NaN` propagando pelo treino.
- **Tudo sai como `Float64`** — o dialeto JDBC do Spark não reconhece os tipos
  sem sinal do ClickHouse e falha com `UNRECOGNIZED_SQL_TYPE` diante de um
  `UInt64`. Se você adicionar um `count()` à query, envolva em `toInt64()`.
- **`LIMIT n BY label`** — amostragem estratificada: `n` linhas de *cada* tipo de
  ataque. Sem isso, um `LIMIT` simples traria só as primeiras tabelas lidas e o
  modelo veria 2 ou 3 famílias de ataque em vez de 18.

As classes são fortemente desbalanceadas: apenas 113.828 linhas BENIGN em
70,4 M — 0,16 % do total. Pegamos todo o tráfego benigno e uma amostra de ataque de tamanho
comparável, chegando a um split de aproximadamente 50/50.

In [4]:
# Colunas de comportamento de fluxo. IPs, portas, timestamp, flow_id e o índice
# original ficam de fora: identificam a captura, não o padrão de tráfego, e o
# modelo decoraria a origem do arquivo em vez de aprender a distinguir ataques.
FEATURES = [
    "protocol", "flow_duration",
    "total_fwd_packets", "total_backward_packets",
    "total_length_of_fwd_packets", "total_length_of_bwd_packets",
    "fwd_packet_length_max", "fwd_packet_length_min",
    "fwd_packet_length_mean", "fwd_packet_length_std",
    "bwd_packet_length_max", "bwd_packet_length_min",
    "bwd_packet_length_mean", "bwd_packet_length_std",
    "flow_bytes_s", "flow_packets_s",
    "flow_iat_mean", "flow_iat_std", "flow_iat_max", "flow_iat_min",
    "fwd_header_length", "bwd_header_length",
    "fwd_packets_s", "bwd_packets_s",
    "min_packet_length", "max_packet_length",
    "packet_length_mean", "packet_length_std",
    "fin_flag_count", "syn_flag_count", "rst_flag_count",
    "psh_flag_count", "ack_flag_count", "urg_flag_count",
    "down_up_ratio", "average_packet_size",
    "init_win_bytes_forward", "init_win_bytes_backward",
    "act_data_pkt_fwd", "min_seg_size_forward",
    "inbound",
]

BENIGN_LIMIT = 100_000   # amostra do tráfego benigno (~114 k linhas no total)
PER_LABEL_LIMIT = 6_000  # linhas por tipo de ataque (18 tipos)

raw_cols = ", ".join(f"`{col}`" for col in FEATURES)
cast_cols = ",\n      ".join(
    f"if(isFinite(toFloat64OrZero(`{col}`)), toFloat64OrZero(`{col}`), 0.) AS `{col}`"
    for col in FEATURES
)

QUERY = f"""
SELECT
      {cast_cols},
      label,
      toFloat64(label != 'BENIGN') AS is_attack
FROM (
    SELECT {raw_cols}, label
    FROM merge(currentDatabase(), '^data')
    WHERE label = 'BENIGN'
    LIMIT {BENIGN_LIMIT}

    UNION ALL

    SELECT {raw_cols}, label
    FROM merge(currentDatabase(), '^data')
    WHERE label != 'BENIGN'
    LIMIT {PER_LABEL_LIMIT} BY label
)
"""

print(f"{len(FEATURES)} features")

41 features


In [5]:
df = (
    spark.read.format("jdbc")
    .option("url", JDBC_URL)
    .option("user", CH_USER)
    .option("password", CH_PASSWORD)
    .option("driver", "com.clickhouse.jdbc.ClickHouseDriver")
    .option("query", QUERY)
    .load()
    .cache()
)

total = df.count()
print(f"{total:,} linhas carregadas do ClickHouse")
df.groupBy("is_attack").count().orderBy("is_attack").show()

198,312 linhas carregadas do ClickHouse


+---------+------+
|is_attack| count|
+---------+------+
|      0.0|100000|
|      1.0| 98312|
+---------+------+



In [6]:
# Cobertura por tipo de ataque: confirma que a estratificação funcionou.
df.groupBy("label").count().orderBy("count", ascending=False).show(25, truncate=False)

+-------------+------+
|label        |count |
+-------------+------+
|BENIGN       |100000|
|DrDoS_UDP    |6000  |
|UDP-lag      |6000  |
|DrDoS_MSSQL  |6000  |
|DrDoS_NTP    |6000  |
|Syn          |6000  |
|DrDoS_SNMP   |6000  |
|DrDoS_SSDP   |6000  |
|NetBIOS      |6000  |
|DrDoS_LDAP   |6000  |
|DrDoS_NetBIOS|6000  |
|Portmap      |6000  |
|UDP          |6000  |
|DrDoS_DNS    |6000  |
|LDAP         |6000  |
|MSSQL        |6000  |
|TFTP         |6000  |
|UDPLag       |1873  |
|WebDDoS      |439   |
+-------------+------+



## 4. Pipeline e treino

`VectorAssembler` → `StandardScaler` → `LogisticRegression`. Regressão logística
é proposital aqui: treina em segundos e os coeficientes são legíveis, o que
serve de baseline honesto antes de partir para algo mais pesado.

In [7]:
from pyspark.ml import Pipeline
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.feature import StandardScaler, VectorAssembler

train_df, test_df = df.randomSplit([0.8, 0.2], seed=42)
print(f"treino: {train_df.count():,}   teste: {test_df.count():,}")

pipeline = Pipeline(stages=[
    VectorAssembler(inputCols=FEATURES, outputCol="features_raw"),
    StandardScaler(inputCol="features_raw", outputCol="features",
                   withMean=True, withStd=True),
    LogisticRegression(featuresCol="features", labelCol="is_attack",
                       maxIter=50, regParam=0.01),
])

model = pipeline.fit(train_df)
print("modelo treinado")

treino: 158,480   teste: 39,832


modelo treinado


## 5. Avaliação

In [8]:
from pyspark.ml.evaluation import (
    BinaryClassificationEvaluator,
    MulticlassClassificationEvaluator,
)

predictions = model.transform(test_df)

auc = BinaryClassificationEvaluator(
    labelCol="is_attack", metricName="areaUnderROC"
).evaluate(predictions)

metrics = {
    name: MulticlassClassificationEvaluator(
        labelCol="is_attack", predictionCol="prediction", metricName=name
    ).evaluate(predictions)
    for name in ("accuracy", "f1", "weightedPrecision", "weightedRecall")
}

print(f"AUC-ROC   {auc:.4f}")
for name, value in metrics.items():
    print(f"{name:<9} {value:.4f}")

AUC-ROC   0.9968
accuracy  0.9919
f1        0.9919
weightedPrecision 0.9919
weightedRecall 0.9919


In [9]:
print("Matriz de confusão (linhas = real, colunas = previsto)\n")
(
    predictions.groupBy("is_attack")
    .pivot("prediction", [0.0, 1.0])
    .count()
    .orderBy("is_attack")
    .show()
)

Matriz de confusão (linhas = real, colunas = previsto)



+---------+-----+-----+
|is_attack|  0.0|  1.0|
+---------+-----+-----+
|      0.0|19994|  175|
|      1.0|  146|19517|
+---------+-----+-----+



In [10]:
# Recall por tipo de ataque: a média geral esconde famílias que o modelo erra.
from pyspark.sql import functions as F

(
    predictions.filter(F.col("label") != "BENIGN")
    .groupBy("label")
    .agg(
        F.count("*").alias("linhas"),
        F.round(F.avg(F.col("prediction")), 4).alias("recall"),
    )
    .orderBy("recall")
    .show(25, truncate=False)
)

+-------------+------+------+
|label        |linhas|recall|
+-------------+------+------+
|WebDDoS      |93    |0.0   |
|UDPLag       |374   |0.9652|
|Portmap      |1211  |0.967 |
|DrDoS_UDP    |1186  |1.0   |
|UDP-lag      |1197  |1.0   |
|DrDoS_NTP    |1222  |1.0   |
|DrDoS_MSSQL  |1183  |1.0   |
|Syn          |1161  |1.0   |
|DrDoS_SNMP   |1207  |1.0   |
|DrDoS_SSDP   |1187  |1.0   |
|NetBIOS      |1170  |1.0   |
|DrDoS_LDAP   |1236  |1.0   |
|DrDoS_NetBIOS|1132  |1.0   |
|DrDoS_DNS    |1204  |1.0   |
|UDP          |1253  |1.0   |
|MSSQL        |1226  |1.0   |
|TFTP         |1193  |1.0   |
|LDAP         |1228  |1.0   |
+-------------+------+------+



In [11]:
# Coeficientes: quais features mais pesam na decisão.
coefficients = model.stages[-1].coefficients.toArray()
ranked = sorted(zip(FEATURES, coefficients), key=lambda kv: abs(kv[1]), reverse=True)

print("Top 15 features por |coeficiente|\n")
for name, value in ranked[:15]:
    print(f"{name:<32} {value:+.4f}")

Top 15 features por |coeficiente|

inbound                          +1.4198
urg_flag_count                   -1.0187
bwd_packet_length_min            -0.7553
protocol                         +0.5873
min_packet_length                +0.5330
fwd_packet_length_min            +0.4912
ack_flag_count                   +0.4887
packet_length_mean               +0.4849
fwd_packet_length_mean           +0.4775
average_packet_size              +0.4611
down_up_ratio                    -0.3785
packet_length_std                -0.2978
init_win_bytes_backward          -0.2773
fwd_packet_length_std            -0.2773
fwd_packet_length_max            +0.2748


## 6. Próximos passos

- **Multiclasse** — trocar `is_attack` por `label` indexado classifica *qual*
  ataque, não só se houve ataque.
- **Modelos de árvore** — `RandomForestClassifier` costuma render bem melhor
  aqui, já que as features de fluxo têm fronteiras não lineares.
- **Camada tipada no ClickHouse** — hoje o raw é todo `String` com
  `ORDER BY tuple()`. Uma tabela derivada com tipos reais e uma chave de
  ordenação útil dispensaria o cast a cada leitura.

In [12]:
spark.stop()